# SPAI Project 1 - Group 4

## `l-diversity` implementation on `Top Down Greedy`for dataset: `CAHOUSING`

First result of baseline model, without any changes made:

Top-down Greedy, cahousing, k=2

* NCP: 0.156
* CAVG after: 1.204
* DM after: 55316
* Runtime: 12.603 s

For comparability of the experiment results, we added `random_seed(42)`. This is the 'official' baseline result for comparison:

- k = 2
- NCP = 0.155
- CAVG = 1.208
- DM = 56764
- Partitions = 9029

### First _l-diversity_ extension

The original TDG algorithm was extended with a distinct _l_-diverstiy check.

For a partition to be considered valid, it must satisfy:
* k-anonymity: partition size >= k
* distinct _l-diversity_: at least _l_ distinct values of the sensitive attribute

For the `cahousing` dataset, `ocean_proximity` was selected as the sensitive attribute.
The first implementation rejected a proposed TDG split whenever one of the resulting child partitions violated l-diversity.

#### Experiment

**Configuration:**

- Dataset: `cahousing`
- Algorithm: Top-down Greedy
- k = 2
- _l_ = 2
- Sensitive attribute: `ocean_proximity`
- Random seed: 42

**Results:**

- NCP: 1.000
- CAVG*: 10320.000
- DM: 426009600
- Runtime: 1.766 s
- Final number of partitions: 1

*The dataset contains 20640 records. With k=2 the final result consists of exactly one equivalence class, thus CAVG becomes:
$$\frac{20640}{1 \cdot 2} = 10320$$

### Observation

The algorithm retained the entire dataset as one equivalence class.
The inital split produced at least one child partition that did not satisfy _l-diversity_. Since the implementation immediately rejected such a split, the parent partition was retained instead.
Although the final partition satisfies 2-diversity, this leads to complete generalization of the QI and therefore very poor data utility.
This indicates that simply rejecting the first invalid TDG split is too restrictive. A better strategy should attempt alternative splits before giving up on partitioning.


**Strategy:**

Replace the current "reject the first invalid split" behavior with a helper that tries several candidate TDG splits and accepts the first one that satisfies k-anonymity and l-diversity.

### After fixing-l-diversity

**Configuration:**

- Dataset: `cahousing`
- Algorithm: Top-down Greedy
- k = 2
- _l_ = 2
- Sensitive attribute: `ocean_proximity`
- Random seed: 42

**Results:**

- NCP: 0.469
- CAVG: 15.426
- DM: 4360456
- Runtime: 17.171 s
- Final number of partitions: 743

In [2]:
import pandas as pd

path = "results/cahousing/topdown/cahousing_anonymized_k2_l2.csv"

df_anon = pd.read_csv(path, sep=";")

qi_columns = [
    "longitude",
    "latitude",
    "housing_median_age",
    "median_income",
    "median_house_value"
]

sensitive_column = "ocean_proximity"

equivalence_classes = df_anon.groupby(qi_columns, dropna=False)

class_sizes= equivalence_classes.size()

diversity = equivalence_classes[sensitive_column].nunique()

print("Number of equivalence classes:", len(class_sizes))
print("Minimum equivalence class size:", class_sizes.min())
print("Minimum l-diversity:", diversity.min())

print("Violations of k=2:", (class_sizes < 2).sum())
print("Violations of l=2:", (diversity < 2).sum())

Number of equivalence classes: 669
Minimum equivalence class size: 2
Minimum l-diversity: 2
Violations of k=2: 0
Violations of l=2: 0
